In [1]:
import os
import sys
import json
import numpy as np
import torch
from pathlib import Path
from skimage.io import imread
import napari
sys.path.append(os.path.abspath(".."))
import data
import models
from data import RangeAnnotationDataset
from models import AutomaticRangeNet
from tifffile import imread, imwrite
from torchvision import transforms
from importlib import reload

In [2]:
import data
import models
reload(data)
from data import RangeAnnotationDataset
reload(models)
from models import AutomaticRangeNet

In [3]:
# Load trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutomaticRangeNet().to(device)
model.load_state_dict(torch.load("../checkpoints/training_set_processed_CD4_nsamp_20_ntile_5_07102025_automatic_range.pt", map_location=device))
model.eval()

/tmp/ipykernel_2934990/121097777.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../checkpoints/training_set_processed_CD4_nsamp_20_nti

AutomaticRangeNet(
  (encoder): Sequential(
    (0): Conv2d(2, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_s

In [17]:
def infer_grid(img_marker, img_dapi, model, patch_size=200, stride=100, device='gpu'):
    model.eval()
    h, w = img_marker.shape
    min_preds = []
    max_preds = []
    coords = []

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            marker_patch = img_marker[y:y+patch_size, x:x+patch_size]
            dapi_patch = img_dapi[y:y+patch_size, x:x+patch_size]
            patch = torch.from_numpy(np.stack([marker_patch, dapi_patch])[None, :, :, :]).float().to(device)

            with torch.no_grad():
                pred_min, pred_max = model(patch).squeeze().tolist()
            
            min_preds.append(pred_min)
            max_preds.append(pred_max)
            coords.append((y, x))
    
    grid_shape = (
        (h - patch_size) // stride + 1,
        (w - patch_size) // stride + 1
    )
    min_grid = np.array(min_preds).reshape(grid_shape)
    max_grid = np.array(max_preds).reshape(grid_shape)
    return min_grid, max_grid, coords

In [ ]:
#imwrite("../data/patch_y_" + str(y) + "_x" + str(x) + "_" + str(pred_min) + "_" + str(pred_max)+ ".tiff", marker_patch.astype(np.float16))


In [23]:
import numpy as np
from scipy.ndimage import gaussian_filter
from skimage.transform import resize

def smooth_predictions(min_grid, max_grid, target_shape, method="gaussian", sigma=1.0):
    if method == "gaussian":
        min_smoothed = gaussian_filter(min_grid, sigma=sigma)
        max_smoothed = gaussian_filter(max_grid, sigma=sigma)
        min_interp = resize(min_smoothed, target_shape, order=1)  # bilinear
        max_interp = resize(max_smoothed, target_shape, order=1)
    elif method == "bicubic":
        min_interp = resize(min_grid, target_shape, order=3)  # bicubic
        max_interp = resize(max_grid, target_shape, order=3)
    else:
        raise ValueError("Method must be 'gaussian' or 'bicubic'")
    return min_interp, max_interp

In [6]:
def normalize_by_range(img, vmin_map, vmax_map):
    norm = (img - vmin_map) / (vmax_map - vmin_map + 1e-8)
    norm = np.clip(norm, 0, 1)
    return norm

In [24]:
def predict_range(marker, dapi, model, device = 'cpu', patch_size=200, stride=100):
    
    # Infer the grid
    min_grid, max_grid, coords = infer_grid(marker, dapi, model, patch_size=patch_size, stride=stride, device=device)
    
    # Smooth the predictions
    min_interp, max_interp = smooth_predictions(min_grid, max_grid, marker.shape, method="gaussian", sigma=1.0)
    
    # Normalize the image using the smoothed min and max grids
    marker_norm = normalize_by_range(marker, min_interp, max_interp)

    return marker_norm, min_interp, max_interp

In [18]:
import random

markers = ["CD4"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
folder_path = "/media/localadmin/EGlab20TB-5/TMA_final_run_01/input_processed/"
samples = random.sample(os.listdir(folder_path), 3)

In [35]:
samples = ["MF008_early_IA_TMA1"]

In [50]:
viewer = napari.Viewer()

In [31]:
int(np.max(dapi_processed.shape)/2)

3814

In [36]:
import time

marker = "CD4"
for sample in samples:

    print(f"Processing sample: {sample}")
    dapi_img = imread("/media/localadmin/EGlab20TB-5/TMA_final_run_01/input_processed/" + sample + "/DAPI.tiff")
    dapi_processed = RangeAnnotationDataset.preprocess_data(dapi_img, percentile=99.99)

    viewer.add_image(dapi_processed, name=f"{sample} DAPI", colormap='bop blue', contrast_limits=(0, 1))


    for stride in [1200, 600, 200]:
        patch_size = 1200
        print(f"Processing with patch size: {patch_size}, stride: {stride}")
        marker_img = imread("/media/localadmin/EGlab20TB-5/TMA_final_run_01/input_processed/" + sample + "/" + marker + ".tiff")
        
        start_time = time.time()
        img_norm, min_interp, max_interp = predict_range(marker_img, dapi_img, model, device=device, patch_size=patch_size, stride=stride)
        end_time = time.time()
        
        print(f"Predicted range for {marker} in {sample} - min: {min_interp.min()}, max: {max_interp.max()}")
        print(f"Time taken for patch size {stride}: {end_time - start_time:.2f} seconds")
        
        viewer.add_image(img_norm, name=f"{sample} - {marker} - stride {stride}", colormap='gray', blending='additive')

        

Processing sample: MF008_early_IA_TMA1
Processing with patch size: 1200, stride: 1200
Predicted range for CD4 in MF008_early_IA_TMA1 - min: 0.5535385154636618, max: 0.9428713324881725
Time taken for patch size 1200: 1.81 seconds
Processing with patch size: 1200, stride: 600
Predicted range for CD4 in MF008_early_IA_TMA1 - min: 0.5360047270773427, max: 0.9712347820724966
Time taken for patch size 600: 3.74 seconds
Processing with patch size: 1200, stride: 200
Predicted range for CD4 in MF008_early_IA_TMA1 - min: 0.49240277984621067, max: 0.9998461475536567
Time taken for patch size 200: 11.99 seconds


In [56]:
sample = "B010A_reg003_X01_Y01_Z01/"
folder_path = "/home/localadmin/GuenovaLab Dropbox/RESEARCH/PEOPLE/Pacome/Tests/Nimbus/data/gold_standard_labelled/codex_colon/fovs//"

In [58]:
os.listdir(folder_path + sample) 

['CD19.ome.tif',
 'CollIV.ome.tif',
 'CD15.ome.tif',
 'HLADR.ome.tif',
 'Synaptophysin.ome.tif',
 'CD68.ome.tif',
 'ITLN1.ome.tif',
 'CD34.ome.tif',
 'aDefensin5.ome.tif',
 'NKG2D.ome.tif',
 'CD8.ome.tif',
 'CD21.ome.tif',
 'CD45RO.ome.tif',
 'CD3.ome.tif',
 'CD206.ome.tif',
 'DRAQ5.ome.tif',
 'SOX9.ome.tif',
 'CD7.ome.tif',
 'CD163.ome.tif',
 'CD25.ome.tif',
 'MUC2.ome.tif',
 'CD49a.ome.tif',
 'CHGA.ome.tif',
 'CD38.ome.tif',
 'CD127.ome.tif',
 'CD44.ome.tif',
 'CD161.ome.tif',
 'Podoplanin.ome.tif',
 'OLFM4.ome.tif',
 'MUC1.ome.tif',
 'CD4.ome.tif',
 'CD56.ome.tif',
 'CD45.ome.tif',
 'CD36.ome.tif',
 'MUC6.ome.tif',
 'CD16.ome.tif',
 'FAP.ome.tif',
 'Ki67.ome.tif',
 'CD69.ome.tif',
 'CD90.ome.tif',
 'CDX2.ome.tif',
 'CD117.ome.tif',
 'CD49f.ome.tif',
 'CD57.ome.tif',
 'aSMA.ome.tif',
 'Vimentin.ome.tif',
 'CD66.ome.tif',
 'CD138.ome.tif',
 'CD31.ome.tif',
 'Cytokeratin.ome.tif',
 'CD123.ome.tif',
 'BCL2.ome.tif',
 'CD11c.ome.tif']

In [ ]:
import os

# Load trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutomaticRangeNet().to(device)
model.load_state_dict(torch.load("../checkpoints/training_set_processed_CD4_nsamp_20_ntile_5_07102025_automatic_range.pt", map_location=device))
model.eval()

dapi_img = imread(folder_path + sample + "/DRAQ5.ome.tif")
dapi_img = RangeAnnotationDataset.preprocess_data(dapi_img, percentile=99.99)
viewer.add_image(dapi_img, name=f"DAPI", colormap='bop blue', contrast_limits=(0, 1))

# List all tiff files in the sample directory excluding DAPI and CD4
marker_files = [
    f for f in os.listdir(folder_path + sample) 
    if f.endswith(".ome.tif") and "DRAQ5" not in f
]

for marker_file in marker_files:
    marker = marker_file.replace(".ome.tif", "")
    print(f"Processing marker: {marker}")

    marker_img = imread(folder_path + sample + "/" + marker_file)
    marker_img = RangeAnnotationDataset.preprocess_data(marker_img, percentile=99.9)

    img_norm, min_interp, max_interp = predict_range(marker_img, dapi_img, model, device=device, patch_size=1200, stride=600)
    
    print(f"Predicted range for {marker} in {sample} - min: {min_interp.min()}, max: {max_interp.max()}")
    
    viewer.add_image(img_norm, name=f"- {marker}", colormap='gray', blending='additive')
    viewer.add_image(marker_img, name=f" - {marker} - RAW", colormap='gray', blending='additive')





/tmp/ipykernel_2934990/1574587214.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../checkpoints/training_set_processed_CD4_nsamp_20_nt

In [17]:
np.unique(img_norm)

array([0.])

In [24]:
imwrite("../data/min_interp.tiff", min_interp.astype(np.float32))
imwrite("../data/max_interp.tiff", max_interp.astype(np.float32))

In [25]:

marker = "CD5"
for sample in samples:

    print(f"Processing sample: {sample}")
    dapi_img = imread("/media/localadmin/EGlab20TB-5/TMA_final_run_01/input/" + sample + "/DAPI.tiff")
    dapi_processed = RangeAnnotationDataset.preprocess_data(dapi_img, percentile=99.99, remove_bright_artifacts = False)

    marker_img = imread("/media/localadmin/EGlab20TB-5/TMA_final_run_01/input/" + sample + "/" + marker + ".tiff")
    img_norm, min_interp, max_interp = predict_range(marker_img, dapi_img, model, device=device, patch_size=200, stride=100)
    print(f"Predicted range for {marker} in {sample} - min: {min_interp.min()}, max: {max_interp.max()}")
    imwrite("../data/" + sample + ".tiff", img_norm.astype(np.float32))

Processing sample: MF155_early_2_IA_TMA2


TypeError: RangeAnnotationDataset.preprocess_data() got an unexpected keyword argument 'remove_bright_artifacts'

In [15]:
imwrite("../data/" + sample + "_marker_processed.tiff", marker_processed.astype(np.float32))


In [12]:
imwrite("../data/" + sample + "_min_interp.tiff", min_interp.astype(np.float32))
imwrite("../data/" + sample + "_max_interp.tiff", max_interp.astype(np.float32))